# ĐỒ ÁN LAB1 - XÂY DỰNG API TÓM TẮT VĂN BẢN TIẾNG VIỆT

**Họ tên:** Hồ Hữu Nghiêm  
**MSSV:** 24120390  

## Giới thiệu
Notebook này trình bày quá trình xây dựng và triển khai một API tóm tắt văn bản tiếng Việt thông qua nền tàng Hugging Face.  
## Nội dung thực hiện
- Cài đặt các thư viện cần thiết
- Khai báo thư viện và cấu hình hệ thống
- Xây dựng schema cho dữ liệu vào/ra
- Đóng gói logic xử lý mô hình theo hướng service
- Xây dựng các API endpoint
- Khởi chạy server và kiểm tra hoạt động cục bộ
- Tạo đường dẫn public để demo API


### Một số lưu ý quan trọng khi chạy mô hình

- Trước tiên cần **kết nối Wi-Fi/Internet ổn định** để notebook có thể tải thư viện và tải mô hình từ Hugging Face.
- Sau khi mở file, nên **ấn “Run all / Chạy tất cả”** để chạy tuần tự toàn bộ các cell, vì notebook đã được tối ưu sẵn theo đúng thứ tự.
- Không nên chạy lẻ từng cell ngay từ đầu nếu chưa chạy các cell cài đặt và khởi tạo môi trường.
- Các phần **kiểm tra kết nối nội bộ và kiểm tra trạng thái hệ thống** đã được đặt sẵn trong **cell 9**.
- URL dùng để **ping/kiểm tra kết nối API nội bộ** nằm tại cell kiểm tra tương ứng trong notebook, chỉ cần mở đúng cell đó và chạy lại để kiểm tra.
- Vì module này dành cho tóm tắt tin tức là chính nên một số cuộc hội thoại hay một số vấn đề khác có thể bị tóm tắt sai
- Nếu sau khi chạy mà chưa ra kết quả, cần kiểm tra lại:
  - đã bật Internet hay chưa
  - đã chạy đầy đủ các cell hay chưa
  - cell kiểm tra kết nối ở **cell 9** đã phản hồi thành công hay chưa
- Nên giữ nguyên thứ tự các cell trong notebook để tránh lỗi do thiếu bước khởi tạo mô hình hoặc khởi tạo API.

## 1. Cài đặt thư viện

Thực thi cell dưới đây để cài đặt các thư viện cần thiết cho mô hình, API và quá trình kiểm thử.

In [ ]:
%%writefile requirements.txt
transformers==4.41.2
torch
sentencepiece
protobuf
fastapi
uvicorn
pydantic
requests
nest_asyncio
pinggy

In [ ]:
!pip install -q -r requirements.txt

## 2. Import thư viện

Cell này khai báo các thư viện phục vụ cho quá trình xây dựng mô hình, triển khai API và chạy server.

In [ ]:
from __future__ import annotations

import json
import textwrap
import threading
import time
import re
from dataclasses import dataclass
from typing import Optional

import nest_asyncio
import requests
import torch
import uvicorn

from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

## 3. Cấu hình mô hình và tham số sinh văn bản

Phần này định nghĩa các cấu hình chính cho mô hình tóm tắt và các tham số dùng trong quá trình sinh kết quả.

In [ ]:
@dataclass(frozen=True)
class ModelConfig:
    model_name: str = "VietAI/vit5-base-vietnews-summarization"
    max_input_length: int = 512


@dataclass(frozen=True)
class GenerationConfig:
    max_length: int = 80
    min_length: int = 20
    num_beams: int = 4
    early_stopping: bool = True
    no_repeat_ngram_size: int = 3
    length_penalty: float = 2.0

## 4. Khai báo schema dữ liệu
Xây dựng các lớp dữ liệu đầu vào và đầu ra để chuẩn hóa request và response của API.

In [ ]:
class SummarizeRequest(BaseModel):
    text: str = Field(
        ...,
        min_length=20,
        description="Văn bản tiếng Việt cần tóm tắt (tối thiểu 20 ký tự)."
    )


class SummarizeResponse(BaseModel):
    model: str
    input: str
    summary: str


class HealthResponse(BaseModel):
    status: str
    model: str
    loaded: bool
    device: str



## 5. Xây dựng lớp dịch vụ xử lý tóm tắt văn bản

Đóng gói toàn bộ quá trình tải tokenizer, mô hình và thực hiện suy luận nhằm tránh lặp mã và tăng tính tổ chức cho chương trình.

In [ ]:
class SummarizationService:
    def __init__(
        self,
        model_config: ModelConfig | None = None,
        generation_config: GenerationConfig | None = None
    ):
        self.model_config = model_config or ModelConfig()
        self.generation_config = generation_config or GenerationConfig()

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self._tokenizer: Optional[AutoTokenizer] = None
        self._model: Optional[AutoModelForSeq2SeqLM] = None
        self._lock = threading.Lock()

    def _load_model(self) -> None:
        if self._tokenizer is not None and self._model is not None:
            return

        with self._lock:
            if self._tokenizer is not None and self._model is not None:
                return

            self._tokenizer = AutoTokenizer.from_pretrained(
                self.model_config.model_name
            )
            self._model = AutoModelForSeq2SeqLM.from_pretrained(
                self.model_config.model_name
            )
            self._model.to(self.device)
            self._model.eval()

    def is_ready(self) -> bool:
        return self._tokenizer is not None and self._model is not None

    def get_model_name(self) -> str:
        return self.model_config.model_name

    def get_device(self) -> str:
        return self.device

    def summarize(self, text: str) -> str:
        cleaned_text = text.strip()

        if not cleaned_text:
            raise ValueError("Văn bản đầu vào không được để trống.")

        if len(cleaned_text.split()) < 10:
            raise ValueError("Văn bản đầu vào quá ngắn để tóm tắt.")

        self._load_model()

        prompt_text = "Tóm tắt: " + cleaned_text

        inputs = self._tokenizer(
            prompt_text,
            return_tensors="pt",
            max_length=self.model_config.max_input_length,
            truncation=True
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            summary_ids = self._model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=self.generation_config.max_length,
                min_length=self.generation_config.min_length,
                num_beams=self.generation_config.num_beams,
                early_stopping=self.generation_config.early_stopping,
                no_repeat_ngram_size=self.generation_config.no_repeat_ngram_size,
                length_penalty=self.generation_config.length_penalty,
                do_sample=False
            )

        summary = self._tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True
        ).strip()

        if not summary:
            raise RuntimeError("Model không tạo được bản tóm tắt.")

        return summary

## 6. Khởi tạo ứng dụng FastAPI

Cell này tạo ứng dụng FastAPI và cấu hình các thông tin cơ bản của hệ thống.

In [ ]:
app = FastAPI(
    title="Vietnamese Text Summarization API",
    description="API tóm tắt văn bản tiếng Việt sử dụng mô hình ViT5 từ Hugging Face.",
    version="1.0.0",
    docs_url="/docs",
    redoc_url="/redoc"
)

## 7. Xây dựng các API endpoint

Phần này định nghĩa các endpoint chính của hệ thống gồm:
- Endpoint kiểm tra trạng thái hoạt động
- Endpoint cung cấp thông tin tổng quan
- Endpoint thực hiện tóm tắt văn bản

In [ ]:
summarization_service = SummarizationService()

@app.get("/")
def root():
    return {
        "message": "Vietnamese Text Summarization API đang hoạt động",
        "description": "API hỗ trợ tóm tắt văn bản tiếng Việt bằng mô hình Hugging Face.",
        "endpoints": {
            "demo": "/demo",
            "docs": "/docs",
            "health": "/health",
            "predict": "/predict"
        },
        "model": summarization_service.get_model_name()
    }


@app.get("/demo", response_class=HTMLResponse)
def demo():
    return f"""
    <!DOCTYPE html>
    <html lang="vi">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Vietnamese Text Summarizer Demo</title>
        <style>
            * {{
                box-sizing: border-box;
            }}

            body {{
                margin: 0;
                font-family: Arial, sans-serif;
                background: linear-gradient(135deg, #eef2ff, #f8fafc);
                padding: 30px 16px;
                color: #1f2937;
            }}

            .container {{
                max-width: 900px;
                margin: 0 auto;
                background: #ffffff;
                padding: 28px;
                border-radius: 18px;
                box-shadow: 0 10px 30px rgba(0, 0, 0, 0.08);
            }}

            h1 {{
                margin: 0 0 8px;
                text-align: center;
                color: #111827;
            }}

            .subtitle {{
                text-align: center;
                color: #6b7280;
                margin-bottom: 24px;
                line-height: 1.6;
            }}

            .info-grid {{
                display: grid;
                grid-template-columns: repeat(auto-fit, minmax(240px, 1fr));
                gap: 12px;
                margin-bottom: 20px;
            }}

            .card {{
                background: #f9fafb;
                border: 1px solid #e5e7eb;
                border-radius: 12px;
                padding: 14px;
            }}

            .card-title {{
                font-size: 13px;
                color: #6b7280;
                margin-bottom: 6px;
            }}

            .card-value {{
                font-size: 15px;
                font-weight: 600;
                word-break: break-word;
            }}

            textarea {{
                width: 100%;
                min-height: 220px;
                padding: 14px;
                border: 1px solid #d1d5db;
                border-radius: 12px;
                font-size: 15px;
                resize: vertical;
                outline: none;
                transition: 0.2s ease;
            }}

            textarea:focus {{
                border-color: #2563eb;
                box-shadow: 0 0 0 3px rgba(37, 99, 235, 0.10);
            }}

            .actions {{
                display: flex;
                flex-wrap: wrap;
                gap: 12px;
                margin-top: 16px;
            }}

            button {{
                border: none;
                border-radius: 10px;
                padding: 12px 18px;
                font-size: 15px;
                font-weight: 600;
                cursor: pointer;
                transition: 0.2s ease;
            }}

            .btn-primary {{
                background: #2563eb;
                color: white;
            }}

            .btn-primary:hover {{
                background: #1d4ed8;
            }}

            .btn-secondary {{
                background: #e5e7eb;
                color: #111827;
            }}

            .btn-secondary:hover {{
                background: #d1d5db;
            }}

            .section {{
                margin-top: 22px;
            }}

            .section h3 {{
                margin-bottom: 10px;
                color: #111827;
            }}

            .result-box {{
                padding: 16px;
                border-radius: 12px;
                background: #f9fafb;
                border: 1px solid #e5e7eb;
                white-space: pre-wrap;
                line-height: 1.7;
                min-height: 70px;
            }}

            .error {{
                margin-top: 12px;
                color: #b91c1c;
                white-space: pre-wrap;
                font-weight: 500;
            }}

            .success {{
                color: #065f46;
            }}

            .footer-note {{
                margin-top: 22px;
                font-size: 14px;
                color: #6b7280;
                line-height: 1.6;
            }}

            code {{
                background: #f3f4f6;
                padding: 2px 6px;
                border-radius: 6px;
            }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Vietnamese Text Summarization</h1>
            <div class="subtitle">
                Giao diện demo cho API tóm tắt văn bản tiếng Việt bằng Hugging Face
            </div>

            <div class="info-grid">
                <div class="card">
                    <div class="card-title">Mô hình</div>
                    <div class="card-value" id="modelName">{summarization_service.get_model_name()}</div>
                </div>
                <div class="card">
                    <div class="card-title">Trạng thái hệ thống</div>
                    <div class="card-value" id="healthStatus">Chưa kiểm tra</div>
                </div>
                <div class="card">
                    <div class="card-title">Thiết bị</div>
                    <div class="card-value" id="deviceInfo">Chưa kiểm tra</div>
                </div>
            </div>

            <div class="actions">
                <button class="btn-secondary" onclick="checkHealth()">Kiểm tra Health</button>
            </div>

            <div class="section">
                <h3>Nhập văn bản cần tóm tắt</h3>
                <textarea id="inputText" placeholder="Nhập hoặc dán văn bản tiếng Việt vào đây..."></textarea>

                <div class="actions">
                    <button class="btn-primary" onclick="summarizeText()">Tóm tắt</button>
                    <button class="btn-secondary" onclick="clearAll()">Xóa</button>
                </div>

                <div id="error" class="error"></div>
            </div>

            <div class="section">
                <h3>Kết quả tóm tắt</h3>
                <div id="output" class="result-box">Chưa có kết quả.</div>
            </div>

            <div class="section">
                <h3>Thông tin phản hồi</h3>
                <div id="meta" class="result-box">Chưa có dữ liệu.</div>
            </div>

            <div class="footer-note">
                API chính vẫn hoạt động ở:
                <code>/</code>,
                <code>/health</code>,
                <code>/predict</code>,
                <code>/docs</code>.
                Trang này chỉ là giao diện demo tại <code>/demo</code>.
            </div>
        </div>

        <script>
            async function checkHealth() {{
                const healthStatus = document.getElementById("healthStatus");
                const deviceInfo = document.getElementById("deviceInfo");
                const modelName = document.getElementById("modelName");

                healthStatus.textContent = "Đang kiểm tra...";
                deviceInfo.textContent = "...";

                try {{
                    const response = await fetch("/health");
                    const data = await response.json();

                    if (!response.ok) {{
                        healthStatus.textContent = "Lỗi";
                        deviceInfo.textContent = "Không xác định";
                        return;
                    }}

                    healthStatus.textContent = data.status + (data.loaded ? " - model đã sẵn sàng" : " - model chưa sẵn sàng");
                    deviceInfo.textContent = data.device || "Không rõ";
                    modelName.textContent = data.model || "Không rõ";
                }} catch (error) {{
                    healthStatus.textContent = "Không kết nối được";
                    deviceInfo.textContent = "Không xác định";
                }}
            }}

            async function summarizeText() {{
                const text = document.getElementById("inputText").value;
                const errorDiv = document.getElementById("error");
                const outputDiv = document.getElementById("output");
                const metaDiv = document.getElementById("meta");

                errorDiv.textContent = "";
                outputDiv.textContent = "Đang xử lý...";
                metaDiv.textContent = "Đang chờ phản hồi...";

                if (!text.trim()) {{
                    errorDiv.textContent = "Vui lòng nhập nội dung trước khi tóm tắt.";
                    outputDiv.textContent = "Chưa có kết quả.";
                    metaDiv.textContent = "Chưa có dữ liệu.";
                    return;
                }}

                try {{
                    const response = await fetch("/predict", {{
                        method: "POST",
                        headers: {{
                            "Content-Type": "application/json"
                        }},
                        body: JSON.stringify({{
                            text: text
                        }})
                    }});

                    const data = await response.json();

                    if (!response.ok) {{
                        outputDiv.textContent = "Không thể tạo bản tóm tắt.";
                        metaDiv.textContent = JSON.stringify(data, null, 2);
                        errorDiv.textContent = data.detail || "Có lỗi xảy ra khi gọi API.";
                        return;
                    }}

                    outputDiv.textContent = data.summary || "Không có kết quả tóm tắt.";
                    metaDiv.textContent = JSON.stringify(data, null, 2);
                }} catch (error) {{
                    outputDiv.textContent = "Không thể tạo bản tóm tắt.";
                    metaDiv.textContent = "Lỗi kết nối.";
                    errorDiv.textContent = "Lỗi kết nối: " + error.message;
                }}
            }}

            function clearAll() {{
                document.getElementById("inputText").value = "";
                document.getElementById("error").textContent = "";
                document.getElementById("output").textContent = "Chưa có kết quả.";
                document.getElementById("meta").textContent = "Chưa có dữ liệu.";
            }}

            window.onload = function() {{
                checkHealth();
            }};
        </script>
    </body>
    </html>
    """


@app.get("/health", response_model=HealthResponse)
def health():
    return HealthResponse(
        status="ok",
        model=summarization_service.get_model_name(),
        loaded=summarization_service.is_ready(),
        device=summarization_service.get_device()
    )


def clean_text(text: str) -> str:
    if not text:
        return ""

    text = re.sub(r"[\n\r\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


@app.post("/predict", response_model=SummarizeResponse)
def predict(request: SummarizeRequest):
    text = clean_text(request.text)

    if not text:
        raise HTTPException(
            status_code=400,
            detail="'text' không được để trống hoặc chỉ chứa khoảng trắng."
        )

    try:
        summary = summarization_service.summarize(text)
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc
    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail=f"Lỗi khi sinh bản tóm tắt từ model: {exc}"
        ) from exc

    return SummarizeResponse(
        model=summarization_service.get_model_name(),
        input=text,
        summary=summary
    )

## 8. Khởi chạy server

Cell này dùng để khởi động API server ngay trong môi trường notebook.

In [ ]:
import asyncio

def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    uvicorn.run(app, host="127.0.0.1", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(3)

print("Server đang chạy tại: http://127.0.0.1:8000")
print("Swagger Docs: http://127.0.0.1:8000/docs")
print("Health Check: http://127.0.0.1:8000/health")

## 9. Kiểm tra hoạt động của API trên môi trường cục bộ

Thực hiện kiểm tra nhanh các endpoint cơ bản để xác nhận server đã chạy đúng.

In [ ]:
base_url = "http://127.0.0.1:8000"

try:
    print("=== ROOT ===")
    root_response = requests.get(f"{base_url}/")
    print("STATUS:", root_response.status_code)
    print(json.dumps(root_response.json(), indent=2, ensure_ascii=False))

    print("\n=== HEALTH ===")
    health_response = requests.get(f"{base_url}/health")
    print("STATUS:", health_response.status_code)
    print(json.dumps(health_response.json(), indent=2, ensure_ascii=False))

    print("\n=== DEMO ===")
    demo_response = requests.get(f"{base_url}/demo")
    print("STATUS:", demo_response.status_code)
    print(demo_response.text[:500])

except requests.exceptions.ConnectionError:
    print("Không kết nối được tới server. Hãy chạy cell khởi động server trước.")

except Exception as e:
    print("Lỗi:", str(e))

## 10. Kiểm thử chức năng tóm tắt văn bản

Sử dụng một đoạn văn bản mẫu để gửi request đến API và quan sát kết quả trả về.

In [ ]:
import requests
import json
import time
import textwrap

payload = {
    "text": """Ngày 3/4, tại TP.HCM, một hội thảo về trí tuệ nhân tạo đã được tổ chức với sự tham gia của nhiều chuyên gia trong và ngoài nước.
Sự kiện nhằm giới thiệu các ứng dụng AI trong giáo dục, y tế và doanh nghiệp, đồng thời tạo cơ hội cho sinh viên tiếp cận với các xu hướng công nghệ mới.
Tại hội thảo, các diễn giả đã chia sẻ về tiềm năng phát triển của AI tại Việt Nam cũng như những thách thức trong việc đào tạo nguồn nhân lực chất lượng cao.
Ban tổ chức cho biết sẽ tiếp tục triển khai nhiều chương trình đào tạo và hợp tác nghiên cứu trong thời gian tới."""
}

try:
    start_time = time.time()
    predict_response = requests.post("http://127.0.0.1:8000/predict", json=payload)
    end_time = time.time()

    print("PREDICT STATUS:", predict_response.status_code)
    print(f"THỜI GIAN PHẢN HỒI: {end_time - start_time:.2f} giây")

    try:
        result = predict_response.json()
    except:
        print("Không parse được JSON:")
        print(predict_response.text)
        result = {}

    if predict_response.status_code == 200:
        print("\n===== MODEL =====")
        print(result.get("model", ""))

        print("\n===== INPUT =====")
        print(result.get("input", ""))

        print("\n===== SUMMARY =====")
        summary_text = result.get("summary") or ""
        print(textwrap.fill(summary_text, width=90))
    else:
        print("\n===== ERROR =====")
        print(json.dumps(result, indent=2, ensure_ascii=False))

except requests.exceptions.ConnectionError:
    print("Chưa kết nối được tới server local. Hãy chạy cell khởi động server trước.")

In [ ]:
bad_payload = {
    "text": "Quá ngắn"
}

try:
    bad_response = requests.post("http://127.0.0.1:8000/predict", json=bad_payload)

    print("BAD REQUEST STATUS:", bad_response.status_code)
    print(json.dumps(bad_response.json(), indent=2, ensure_ascii=False))

except requests.exceptions.ConnectionError:
    print("Chưa kết nối được server local.")

## 11. Tạo đường dẫn public để trình diễn

Tạo đường dẫn public thông qua Pinggy nhằm phục vụ việc truy cập và demo API từ bên ngoài.

In [ ]:
import pinggy
try:
    tunnel = pinggy.start_tunnel(forwardto="localhost:8000")
    public_url = tunnel.urls[0]

    print("Public URL:", public_url)
    print("Demo URL:", f"{public_url}/demo")
    print("Docs URL:", f"{public_url}/docs")
    print("Health URL:", f"{public_url}/health")
    print("Predict URL:", f"{public_url}/predict")

except Exception as e:
    public_url = None
    print("Không tạo được public URL:", e)

## 12. Kiểm tra API thông qua đường dẫn public

Thực hiện kiểm tra lại endpoint sau khi tạo public URL để xác nhận hệ thống có thể truy cập từ bên ngoài.

In [ ]:
if public_url is None:
    print("Chưa có public URL để kiểm thử.")
else:
    try:
        public_health = requests.get(f"{public_url}/health")
        print("PUBLIC HEALTH STATUS:", public_health.status_code)
        print(json.dumps(public_health.json(), indent=2, ensure_ascii=False))

        public_payload = {
            "text": """Ngày 3/4, tại TP.HCM, một hội thảo về trí tuệ nhân tạo đã được tổ chức với sự tham gia của nhiều chuyên gia trong và ngoài nước.
Sự kiện nhằm giới thiệu các ứng dụng AI trong giáo dục, y tế và doanh nghiệp, đồng thời tạo cơ hội cho sinh viên tiếp cận với các xu hướng công nghệ mới.
Tại hội thảo, các diễn giả đã chia sẻ về tiềm năng phát triển của AI tại Việt Nam cũng như những thách thức trong việc đào tạo nguồn nhân lực chất lượng cao.
Ban tổ chức cho biết sẽ tiếp tục triển khai nhiều chương trình đào tạo và hợp tác nghiên cứu trong thời gian tới."""
        }

        public_predict = requests.post(f"{public_url}/predict", json=public_payload)
        print("\nPUBLIC PREDICT STATUS:", public_predict.status_code)

        result = public_predict.json()

        if public_predict.status_code == 200:
            print("\n===== INPUT =====")
            print(result.get("input", ""))

            print("\n===== SUMMARY =====")
            print(textwrap.fill(result.get("summary", ""), width=90))
        else:
            print(json.dumps(result, indent=2, ensure_ascii=False))

    except requests.exceptions.RequestException as e:
        print("Lỗi khi kiểm thử public API:", e)

## Kết luận

### Một số thông tin cơ bản về mô hình

- **Loại mô hình:** Mô hình học sâu Seq2Seq cho bài toán tóm tắt văn bản.
- **Kiến trúc sử dụng:** ViT5 / T5 cho tiếng Việt.
- **Tên mô hình cụ thể:** `VietAI/vit5-base-vietnews-summarization`
- **Đường dẫn:** `huggingface.co/VietAI/vit5-base-vietnews-summarization`

### Mức độ đáp ứng các yêu cầu

- **GET /**: trả về thông tin giới thiệu ngắn gọn về hệ thống hoặc mô tả chức năng của API.
- **GET /health**: kiểm tra trạng thái hoạt động của hệ thống.
- **POST /predict**: nhận dữ liệu đầu vào từ người dùng, gọi mô hình Hugging Face để sinh bản tóm tắt, và trả kết quả dưới dạng JSON.

### Một số yêu cầu khác

- Có xử lý ngoại lệ cho dữ liệu đầu vào.
- Có kiểm tra ràng buộc điều kiện đầu vào và đầu ra.
- Kết quả trả về rõ ràng dưới dạng JSON.

### Tổng kết

Đồ án đã xây dựng thành công một API tóm tắt văn bản tiếng Việt sử dụng mô hình `VietAI/vit5-base-vietnews-summarization` từ Hugging Face. Hệ thống đáp ứng được các chức năng cơ bản như kiểm tra trạng thái hoạt động, tiếp nhận dữ liệu đầu vào, xử lý bằng mô hình học sâu và trả kết quả tóm tắt dưới định dạng JSON.